<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/ERA5_Daily_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rioxarray

In [ ]:
!pip install --upgrade xee

In [ ]:
!pip install -U geemap

In [ ]:
import rioxarray
import ee
import geemap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
import xee
import geopandas as gpd


In [ ]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
              opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
map = geemap.Map()
map

In [ ]:
point = map.draw_last_feature.geometry()
point

In [ ]:
border = (
    ee.FeatureCollection("FAO/GAUL/2015/level1")
    .filterBounds(point)
)

In [ ]:
map.addLayer(border)

In [ ]:
vec = geemap.ee_to_gdf(border)
vec

In [ ]:
era = (
    ee.ImageCollection("ECMWF/ERA5/DAILY")
    .filterDate("2000-01-01", "2000-01-30")
    .select("mean_2m_air_temperature")
)
era

In [ ]:
from jinja2.nodes import ExprStmt
ds = xr.open_dataset(
    era,
    engine = "ee",
    crs = "EPSG: 4326",
    geometry = border.geometry(),
    scale = 0.00108
)

In [ ]:
ds

In [ ]:
ds.isel(time = 10).mean_2m_air_temperature.plot(
    x = "lon",
    y = "lat",
    robust = True,
)

In [ ]:
import geopandas as pgd

In [ ]:
vec.plot()

In [ ]:
vec.crs

In [ ]:
vec.geometry

In [ ]:
ds_rename = ds.rename({"lon": "x", "lat": "y"})


In [ ]:
ds_clip = ds_rename.rio.clip(vec.geometry, vec.crs)

In [ ]:
import pandas as pd


In [ ]:
ds_clip.isel(time = 10).mean_2m_air_temperature.plot.contourf(
    x = "x",
    y = "y",
    robust = True,
    levels = 25,
    cmap = "turbo_r"
)
plt.tight_layout(rect=[0.02, 0.05, 0.098, 0.95])
plt.gca().set_aspect("equal")
plt.show()

In [ ]:
ds_clip["mean_2m_air_temperature_celsius"] = ds_clip["mean_2m_air_temperature"] - 273.15

In [ ]:
ds_clip.isel(time = 10).mean_2m_air_temperature_celsius.plot.contourf(
    x = "x",
    y = "y",
    robust = True,
    levels = 25,
    cmap = "turbo_r"

)
plt.tight_layout(rect=[0.02, 0.05, 0.98, 0.95])
plt.gca().set_aspect("equal")
plt.show()